# NBA Comp Viz — Colab harvest runner
Runs the full harvest chain (build → segment → matchups → align → join → credit)
on a Colab GPU (~4-6× faster than local MPS). The repo is self-contained: code,
model weights, PBP data and the games registry all clone from GitHub.

**Runtime → Change runtime type → GPU (T4)** before running.

Resume-safe: `data/harvest/status.json` is kept on Drive, so a killed session
picks up where it left off — just Run All again.

In [ ]:
#@title 1 · Clone/UPDATE repo + PURGE stale modules + deps + PREFLIGHT
%cd /content
import os, sys, shutil
if not os.path.exists("/content/repo"):
    !git clone --depth 1 https://github.com/lmcnulty7/NBA-Comp-Viz-V3.git /content/repo
%cd /content/repo
!git fetch origin && git reset --hard origin/main && git log --oneline -1
shutil.rmtree("/content/repo/repo", ignore_errors=True)   # nested clones from old sessions
# purge cached project modules — a long-lived kernel otherwise keeps executing
# OLD code from previous runs no matter what git does (run-5 postmortem)
_OURS = ("config", "gate", "detect", "court", "clock_reader", "fetch_pbp",
         "align_outcomes", "harvest_driver", "tier2_join", "tier2_credit",
         "segment_possessions", "matchup_metrics", "build_trajectories")
for m in [m for m in sys.modules if m.split(".")[0] in _OURS]:
    del sys.modules[m]
sys.path = [p for p in sys.path if "/repo" not in p or p == "/content/repo"]
if "/content/repo" not in sys.path:
    sys.path.insert(0, "/content/repo")
!pip -q install ultralytics easyocr yt-dlp 2>&1 | tail -1
import torch
print("device:", "cuda" if torch.cuda.is_available() else "CPU ONLY — enable GPU runtime!")
REQUIRED = ["models/trained_head_coefs.npz", "models/thresholds.json",
            "models/player_detector.pt", "models/court_grid_snapped.pt",
            "models/court_line_seg.pt", "data/harvest/games.json"]
missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, f"MISSING RUNTIME FILES: {missing}"
import config
assert config.__file__.startswith("/content/repo/"), \
    f"STALE MODULE loaded from {config.__file__} — Restart the kernel, then Run All"
import numpy as np, json as _j
from gate.backbones import get_backbone
from gate.trained_head import TrainedHeadGate
import gate.trained_head as _th
assert _th.__file__.startswith("/content/repo/"), f"stale gate module: {_th.__file__} — restart kernel"
_g = TrainedHeadGate.load(config.HEAD_PATH,
                          backbone=get_backbone("clip", config.get_device()),
                          threshold=_j.loads(config.THRESHOLDS_PATH.read_text())["trained"])
assert _g.meta.get("src") == "npz", "gate loaded the sklearn pickle, not the npz — stale code?"
_s = _g.score(np.random.randint(0, 255, (720, 1280, 3), np.uint8))
print(f"preflight OK — files present, gate scores ({_s:.3f} on noise), head src: npz")

In [ ]:
#@title 2 · Storage: Drive-live persistence (true mid-run resume)
import os
PERSIST = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = "/content/drive/MyDrive/nba_harvest"
except Exception as e:
    print("Drive mount unavailable through this client:", e)
    PERSIST = "/content/nba_harvest"
    print("→ VM-local only: a disconnect LOSES all progress. Avoid for long runs.")
for d in ("video", "tracking", "results"):
    os.makedirs(f"{PERSIST}/{d}", exist_ok=True)
os.makedirs("data/harvest", exist_ok=True)
# LIVE persistence: videos, per-stage status, and section outputs all write
# straight to Drive — a disconnect mid-run costs at most one in-flight stage.
os.system(f"ln -sfn {PERSIST}/video data/harvest/video")
os.system(f"cp -rn data/tracking/. {PERSIST}/tracking/ 2>/dev/null")   # repo files (backups etc.)
os.system(f"rm -rf data/tracking && ln -sfn {PERSIST}/tracking data/tracking")
if not os.path.exists(f"{PERSIST}/status.json"):
    open(f"{PERSIST}/status.json", "w").write("{}")
os.system(f"ln -sf {PERSIST}/status.json data/harvest/status.json")
print("persist dir:", PERSIST, "(status + outputs live on it)")

In [ ]:
#@title 3 · Download games (H.264 ONLY — Colab cv2 can't decode AV1)
# Run-6 postmortem: the old selector (bv*[height<=720]) let YouTube serve AV1;
# Colab's OpenCV has no AV1 decoder, so every frame read failed ([av1] Get
# current frame error, 0 successful reads) while metadata looked fine. Pin the
# selector to avc1 and VERIFY THE ARTIFACT's codec with ffprobe — cached AV1
# files on Drive are deleted and re-fetched.
GAMES = ["gsw_cle_2017f_g5", "gsw_sac_klay37", "gsw_nyk_curry54",
         "gsw_mia_2017", "gsw_splash62", "gsw_hou_duel"]
import json, subprocess, os, glob
reg = json.load(open("data/harvest/games.json"))

def vinfo(p):
    r = subprocess.run(["ffprobe", "-v", "quiet", "-select_streams", "v:0",
                        "-show_entries", "stream=codec_name,height",
                        "-of", "csv=p=0", p], capture_output=True, text=True)
    return (r.stdout or "").strip()          # e.g. "h264,720"

for t in GAMES:
    dst = f"data/harvest/video/{t}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 1e8:
        info = vinfo(dst)
        if info.startswith("h264"):
            print(t, "cached OK", info); continue
        print(t, f"cached but [{info}] — deleting + re-downloading as avc1")
        os.remove(dst)
        for s in glob.glob(f"data/harvest/video/{t}_s*.mp4"):
            os.remove(s)                      # sections split from the AV1 file
    vid = reg[t]["video_id"]
    r = subprocess.run(["yt-dlp", "-f",
                        "bv*[vcodec^=avc1][height<=720][height>=480]+ba/"
                        "b[vcodec^=avc1][height<=720]",
                        "--merge-output-format", "mp4", "-o", dst, "-q",
                        "--no-warnings", f"https://www.youtube.com/watch?v={vid}"])
    info = vinfo(dst) if os.path.exists(dst) else "missing"
    if r.returncode == 0 and info.startswith("h264"):
        print(t, "OK", info)
    else:
        print(t, f"FAILED [{info}] — no avc1 <=720p stream? Do NOT run this game; "
              "report the tag. (If YouTube blocks Colab IPs, upload an h264 mp4 "
              "from your laptop to Drive:nba_harvest/video/ instead.)")


In [ ]:
#@title 4 · Run the harvest queue (videos LOCALIZED off Drive first)
# cv2 frame-seeking over the Drive FUSE mount fails silently (run-2 postmortem:
# every build read 0 frames and wrote {}). Videos are COPIED to VM-local disk
# for processing; Drive stays the durable download cache. Status + outputs stay
# on Drive (small JSON writes are FUSE-safe), so resume still works — a fresh
# VM just re-copies videos (~1-2 min/game).
import os, shutil, subprocess, glob
LOCAL_V = "/content/video_local"
os.makedirs(LOCAL_V, exist_ok=True)
for f in glob.glob(f"{PERSIST}/video/*.mp4"):
    dst = f"{LOCAL_V}/{os.path.basename(f)}"
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f)):
        print("localizing", os.path.basename(f))
        shutil.copy(f, dst)
        # game file changed (e.g. AV1 -> h264 re-download): stale local sections
        # split from the old file must go too, or split_sections keeps them
        tag = os.path.basename(f)[:-4]
        for s in glob.glob(f"{LOCAL_V}/{tag}_s*.mp4"):
            os.remove(s)
# codec sweep — AV1 must never reach a build again (run-6 postmortem)
def _vcodec(p):
    r = subprocess.run(["ffprobe", "-v", "quiet", "-select_streams", "v:0",
                        "-show_entries", "stream=codec_name", "-of", "csv=p=0", p],
                       capture_output=True, text=True)
    return (r.stdout or "").strip()
for f in sorted(glob.glob(f"{LOCAL_V}/*.mp4")):
    c = _vcodec(f)
    if c != "h264":
        raise RuntimeError(f"{os.path.basename(f)} is [{c}], not h264 — "
                           "re-run cell 3 (it re-fetches avc1).")
os.system(f"ln -sfn {LOCAL_V} data/harvest/video")
r = subprocess.run(["python", "harvest_driver.py", "--games"] + GAMES + ["--no-align"])
print("driver exit", r.returncode)
# show any failed units WITH their reasons — no more silent no-ops
import json as _j
st = _j.load(open("data/harvest/status.json"))
for c, u in sorted(st.items()):
    for stage, rec in u.items():
        if rec.get("state") != "ok":
            print("FAILED", c, stage, "→", (rec.get("tail") or "")[:150])


In [ ]:
#@title 5 · Align + join + credit (CPU-light, GPU OCR)
import json, subprocess
st = json.load(open("data/harvest/status.json"))
secs = sorted(c for c, u in st.items() if "_s" in c and u.get("segment", {}).get("state") == "ok"
              and any(c.startswith(t) for t in GAMES))
print(len(secs), "sections to align")
subprocess.run(["python", "align_outcomes.py", "--clips"] + secs)
for c in secs:
    subprocess.run(["python", "matchup_metrics.py", "--clip", c, "--no-video"],
                   capture_output=True)
subprocess.run(["python", "tier2_join.py"])
subprocess.run(["python", "tier2_credit.py"])

In [ ]:
#@title 6 · Package results
import os
os.system(f'cd {PERSIST} && zip -qr results/night3_tracking.zip tracking -i "tracking/gsw_*"')
os.system(f"zip -qr {PERSIST}/results/night3_pbp.zip data/pbp")
print("results in:", f"{PERSIST}/results")
if PERSIST.startswith("/content/nba_harvest"):
    print("NO DRIVE — download both zips via the file browser BEFORE the VM recycles.")
else:
    print("on Drive: nba_harvest/results/ — everything already persisted live anyway.")